In [89]:
import torch 
import yaml
from pathlib import Path
import sys
sys.path.append('..')
from utils.inference.fno_inference_utils import fno_forecast
from datasets.heat_dataset import HeatFNODataset
from torch.utils.data import DataLoader
from models.forecasting.FNO import FNO
from models.model_utils.nn_helpers.ffn import FFN
from tqdm import tqdm

# LOADING MODEL PARAMETERS  
model_folder_path = '../checkpoints/fno_heat/20260629_2250'
yaml_files = list(Path(model_folder_path).glob("*.yaml"))
model_cfg_path = max(yaml_files, key=lambda p: p.stat().st_mtime)
with open(model_cfg_path, "r") as file:
    model_cfg = yaml.safe_load(file)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
    )

def load_fno_model_(cfg, model_path, device):
    # --- Model ---
    P_FFN = FFN(layer_sizes=cfg['num_p_layers'], activation=cfg['p_activations'])
    Q_FFN = FFN(layer_sizes=cfg['num_q_layers'], activation=cfg['q_activations'])

    model = FNO(d_v=cfg['projection_dim'],
                num_fourier_layers=cfg['num_fourier_layers'],
                num_fourier_modes=cfg['num_fourier_modes'],
                P=P_FFN,
                Q=Q_FFN,
                optimiser=cfg['optimiser'],
                learning_rate=cfg['learning_rate'])

    model.load_state_dict(torch.load(model_path, map_location=device)['state_dict'])
    model.to(device)
    model.eval()
    return model

ckpt_files = list(Path(model_folder_path).glob("*.ckpt"))
model_path = str(max(ckpt_files, key=lambda p: p.stat().st_mtime))

fno_model = load_fno_model_(cfg=model_cfg, model_path=model_path, device=device)

In [95]:
def batch_forecast(X, model, num_steps):
    # X has shape (B, C , H, W)

    # y_hat should have shape (B, 1, H, W)
    forecasts = []
    for i in range(num_steps):
        y_hat = model(X)
        forecasts.append(y_hat)
        print(y_hat.shape)
        X[:, 0, :,  :] = y_hat[:, 0, :, :]

    return torch.concat(forecasts, dim = 1)


In [96]:
# LOAD INPUT DATA
data_path = '../data/test_data/heat_equation_m64_h0_minmax_N200.pt'
X_test = torch.load(data_path)

# Inference
batch_size = 10
batch_indices = [idx*batch_size for idx in range(0, X_test['X'].shape[0]//batch_size + 1)]
num_forecast_steps = 199

A = []
for i in X_test['a']:
    single_a = torch.full_like(X_test['X'][0, 0].unsqueeze(0), i)
    A.append(single_a)

A = torch.concat(A)

for idx in tqdm(range(len(batch_indices))):

    if idx == len(batch_indices) - 1: 
        batch_input_x = X_test['X'][batch_indices[idx]: 0]
        batch_input_a = A[batch_indices[idx]: ]
        X = torch.stack([batch_input_x] + [batch_input_a], dim=1).to(device=device)

    else:
        batch_input_x = X_test['X'][batch_indices[idx] : batch_indices[idx+1], 0]
        batch_input_a = A[batch_indices[idx] : batch_indices[idx+1]]
        X = torch.stack([batch_input_x] + [batch_input_a], dim=1).to(device=device)

    single_batch = batch_forecast(X, fno_model, num_steps = 20) 

    break

  0%|          | 0/21 [00:00<?, ?it/s]

torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])


  0%|          | 0/21 [00:00<?, ?it/s]

torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])


In [97]:
single_batch.shape

torch.Size([10, 20, 64, 64])

In [87]:
X_test['X'].shape

torch.Size([200, 200, 64, 64])